In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd


df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')


In [ ]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill missing values
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])

# Check if the missing values are handled
print(df.isnull().sum())


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encoding categorical variables
label_encoders = {}
for column in categorical_cols:
    label_encoders[column] = LabelEncoder()
    df[column] = label_encoders[column].fit_transform(df[column])

# Check the first few rows to see the encoded categorical columns
df.head()


In [ ]:
from sklearn.model_selection import train_test_split

# Define the features (X) and the target (y)
X = df.drop('Attrition', axis=1)  # Replace 'Attrition' with your target column
y = df['Attrition']

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train a Decision Tree model
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

# Make predictions and evaluate the model
y_pred_dt = dt_model.predict(X_test)
print("Decision Tree Accuracy: ", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


In [ ]:
from sklearn.svm import SVC

# Train an SVM model
svm_model = SVC()
svm_model.fit(X_train, y_train)

# Make predictions and evaluate the model
y_pred_svm = svm_model.predict(X_test)
print("SVM Accuracy: ", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


In [ ]:
from sklearn.neural_network import MLPClassifier

# Train a Neural Network (MLP) model
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300)
mlp_model.fit(X_train, y_train)

# Make predictions and evaluate the model
y_pred_mlp = mlp_model.predict(X_test)
print("Neural Network Accuracy: ", accuracy_score(y_test, y_pred_mlp))
print(classification_report(y_test, y_pred_mlp))


In [ ]:
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation on the Decision Tree model
cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5)
print(f"Decision Tree Cross-Validation Scores: {cv_scores_dt}")
print(f"Mean Cross-Validation Score: {cv_scores_dt.mean()}")

# Perform 5-fold cross-validation on the SVM model
cv_scores_svm = cross_val_score(svm_model, X_train, y_train, cv=5)
print(f"SVM Cross-Validation Scores: {cv_scores_svm}")
print(f"Mean Cross-Validation Score: {cv_scores_svm.mean()}")

# Perform 5-fold cross-validation on the Neural Network model
cv_scores_mlp = cross_val_score(mlp_model, X_train, y_train, cv=5)
print(f"Neural Network Cross-Validation Scores: {cv_scores_mlp}")
print(f"Mean Cross-Validation Score: {cv_scores_mlp.mean()}")


In [ ]:
from sklearn.model_selection import GridSearchCV

# Decision Tree Hyperparameter Tuning
dt_params = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}
grid_dt = GridSearchCV(DecisionTreeClassifier(), dt_params, cv=5)
grid_dt.fit(X_train, y_train)
print(f"Best Params for Decision Tree: {grid_dt.best_params_}")
print(f"Best Score: {grid_dt.best_score_}")

# SVM Hyperparameter Tuning
svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
grid_svm = GridSearchCV(SVC(), svm_params, cv=5)
grid_svm.fit(X_train, y_train)
print(f"Best Params for SVM: {grid_svm.best_params_}")
print(f"Best Score: {grid_svm.best_score_}")


In [ ]:
from sklearn.metrics import confusion_matrix, plot_confusion_matrix
import matplotlib.pyplot as plt

# Confusion Matrix for the best-performing model (example: Neural Network)
conf_matrix = confusion_matrix(y_test, y_pred_mlp)
print("Confusion Matrix for Neural Network:\n", conf_matrix)

# Plot confusion matrix
plot_confusion_matrix(mlp_model, X_test, y_test, cmap=plt.cm.Blues)
plt.title("Confusion Matrix for Neural Network")
plt.show()


In [ ]:
!pip install Flask


In [ ]:
from flask import Flask, request, jsonify
import joblib

# Save the trained model to a file
joblib.dump(mlp_model, 'mlp_model.pkl')

# Load the model (for inference)
model = joblib.load('mlp_model.pkl')

app = Flask(__name__)

# API for predicting employee attrition
@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    prediction = model.predict([data['features']])  # Assuming 'features' is a list of employee attributes
    return jsonify({'prediction': int(prediction[0])})

if __name__ == '__main__':
    app.run(port=5000, debug=True)


In [ ]:
curl -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" -d '{"features": [0, 5, 1, 40, 20000, 5]}'
